# Search Evaluation
Now that we have ground truth data, we can evaluate how well our search
retrieves the correct documents.

For each question in our ground truth dataset, we run search. Then we
check whether the results include the correct document.

## Setting up search

Create a new notebook for search evaluation. We'll use the ground truth
CSV from the previous lesson and set up the search index here.

For search evaluation, we only need the search part of the RAG
pipeline. We don't need to call the LLM yet.

Load the ground truth file from the previous notebook:


In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv")

In [3]:
df_ground_truth.head()

,question,document
0,I just found this course late — can I still jo...,74eb249bbf
1,"If I start the course after it begins, do I st...",74eb249bbf
2,Is it okay to enroll now even though I missed ...,74eb249bbf
3,What do I need to do to be eligible for the co...,74eb249bbf
4,"Can I still take part in the course now, or is...",74eb249bbf


In [ ]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'document': '489dd1c9d9'}

Wrap the search call in a function called `text_search`. The name is
deliberate. Later we'll write `vector_search` or a hybrid version and
run the exact same evaluation on it. Everything downstream only needs a
function that takes a query and returns results, so we can swap one for
another. That mirrors how RAG works: the retrieval step doesn't care
which search function sits behind it.

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [ ]:
boost = {'question' : 3.0}

index.search(
    'What is the course about?',
    num_results=5,
    boost_dict=boost
)

[{'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.'},
 {'id': '489dd1c9d9

## Collecting relevance data

Start with one ground truth record:

In [8]:
q = ground_truth[0]
q

{'question': 'I just found this course late — can I still join and follow along?',
 'document': '74eb249bbf'}

Run search for this question:


In [62]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
doc_id = q["document"]
results = text_search(query=q["question"])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch

First, compare the retrieved document IDs with the correct document ID:


In [13]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
04919992b3 == 74eb249bbf: False
69d122f12e == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False


Then turn this comparison into a relevance list. In this lesson,
relevance means whether a retrieved document is the correct document
for this question.


In [14]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

This gives a list of `0` and `1` values. `1` means the retrieved
document has the same ID as the correct document.

Put this logic into a function:

In [15]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

The correct document was the first search result.

Here are two more examples from the generated ground truth data.

For this question:

In [ ]:
q = ground_truth[10]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I find the course materials, videos, and homework deadlines?


[0, 0, 1, 0, 0]

Also, we can obtain the result according to our relevance, getting something like a matrix in each position:
<br>&emsp;Q_1   : [1, 0, 0, 0]
<br>&emsp;Q_2   : [0, 1, 0, 0]
<br>&emsp;Q_3   : [0, 0, 1, 0]
<br>&emsp;Q_4   : [0, 0, 0, 1]
<br>&emsp;Q_n   : [*, *, *, *]

In [29]:
q = ground_truth[16]
print(q["question"])
compute_relevance_text(q)

Where do I find the course materials, videos, and homework deadlines?


[0, 0, 1, 0, 0]

The correct document was found at the first position again.

Now do the same thing for all ground truth questions:


In [30]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

Call it for the first 15 ground truth questions:


In [31]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [32]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0]]

<details> 
<summary>For the data we prepared on May 29, 2026, this gives (click to show)</summary>

```python
[
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]
```
</details>

Each entry in `relevance_total_text` is a relevance list. This is
enough to check that the function works before we run it for the full
dataset.

Next, make the relevance functions generic. We start with text search,
but later we may want to evaluate vector search, hybrid search, or
another retrieval method. The relevance logic is the same. Only the
search function changes.

In [33]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

The total relevance function gets a `search_function` too.

We need to provide it explicitly:

In [34]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

Use it with `text_search` on the same sample:

In [35]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0]]

Now run it for all ground truth questions:

In [36]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]

In [40]:
relevance_total[:20]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0]]

Now we can represent search results as relevance lists. In the next
lesson, we'll turn these lists into metrics: Hit Rate and MRR.

# Search Evaluation Metrics



In the previous lesson, we computed relevance lists for search results.
We can turn those lists into metrics.

## Hit Rate

Hit Rate (also called Recall@k) measures the fraction of queries where
the correct document appears anywhere in the results:

<details>
<summary>Click to show results</summary>

```python
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]
```
</details>

Each line is one query. If a line contains `1`, search found the
correct document somewhere in the top 5 results. If the line contains
only zeros, search did not find the correct document.

In our setup, each query has one correct document, so Hit Rate and
Recall@k mean the same thing.

Let's calculate it:

In [47]:
cnt = 0

for line in relevance_total_text:
    if 1 in line:
        cnt = cnt + 1

cnt

12

There are 12 hits. The example has 15 queries.

The Hit Rate is:

In [46]:
cnt / relevance_total_text.__len__()

0.8

Put the same logic into a function:

In [48]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
print(hit_rate(relevance_total))
print(relevance_total.__len__())

0.8388349514563107
515


## Mean Reciprocal Rank (MRR)

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct
document:

- position 1: score is 1.0
- position 2: score is 0.5
- position 3: score is 0.333
- not found: score is 0

In the example, most hits are at the first position. Some hits are
lower in the list.

Look at that line:
```python
example[1]
# [0, 1, 0, 0, 0]
```
For this line, the score is `1/2` because the correct document is at
position 2.

Let's calculate MRR:

In [55]:
total_score = 0.0

for line in relevance_total_text:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

10.666666666666666

The total score is `12.333333333333334`. We use `rank + 1` because
Python counts positions from zero. The first position should score
`1/1`, and without the `+ 1` we'd divide by zero.

Divide it by the number of queries:

In [57]:
total_score / len(relevance_total_text)
# 0.822 | other data values

0.711111111111111

MRR is the average of these scores across all queries. It rewards
systems that put the correct document near the top.

Hit Rate is the upper bound for MRR. In practice, MRR is usually
smaller because some correct documents are found below the first
position.

Put the same logic into a function:

In [58]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

Check it on the same example:


In [ ]:
mrr(relevance_total_text)

0.711111111111111

In [60]:
mrr(relevance_total)

0.7009708737864077

## Putting it together

Wrap the metrics in a reusable evaluation function:

In [61]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

We can evaluate any search function:


In [63]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.8388349514563107, 'mrr': 0.7009708737864077}

Search metrics tell us whether retrieval works. Next, we'll use these metrics to tune the search parameters.

## Interpreting the metrics

A few things to keep in mind when reading these numbers:

Our ground truth assumes only one relevant document per query. In
practice, other retrieved documents might also be relevant. A 50% hit
rate does not mean that half the results are useless. It means the
document we generated the question from did not appear in the top
results for half the queries. Other relevant documents may still be
there.

With synthetic data, the generated questions can be too close to the
original FAQ text. This inflates hit rate and MRR. If you see numbers
above 95%, treat them with caution and check whether the questions are
realistic enough.

Good thresholds depend on your use case. A 50% hit rate is acceptable
for some applications, while others need 90% or higher. The right
number depends on how much the downstream LLM can compensate for
imperfect retrieval. It also depends on user tolerance for wrong
answers.

Look at the system holistically. A high MRR means the relevant document
is near the top, which helps the LLM focus on the right context. A low
MRR with a high hit rate means the document is there, but buried under
less relevant results.

> Saving data to use in other notebook

In [64]:
%store relevance_total

Stored 'relevance_total' (list)


In [65]:
%store relevance_total_text

Stored 'relevance_total_text' (list)


# Search Parameter Tuning
In the previous lesson, we defined Hit Rate, MRR, and the `evaluate`
function. Now we can use them to tune search parameters.

Instead of guessing which settings are better, we measure them on the
ground truth dataset.

So far we've boosted `question` to 3.0. The idea was that a query should
match the FAQ question. That kind of match should count for more than
matching the answer text. It sounds reasonable. But it's a guess, and now
we can check it against data instead of trusting it.

This is the main benefit of offline evaluation. We change one parameter,
run the same questions again, and see whether the metric moves. The
dataset stays fixed, so the comparison is fair.

## Trying different boosts

Start with a search function where the question boost is configurable:

In [77]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [75]:
evaluate(
    ground_truth,
    search_boost
)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.8970873786407767, 'mrr': 0.7728802588996762}

Evaluate several boost values:


In [78]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/515 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8854368932038835, 'mrr': 0.7848220064724918}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8970873786407767, 'mrr': 0.7728802588996762}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8388349514563107, 'mrr': 0.7009708737864077}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8116504854368932, 'mrr': 0.6691262135922329}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7805825242718447, 'mrr': 0.6458576051779934}


For the data we prepared on May 29, 2026, this gives:

```python
boost=0.5: {'hit_rate': 0.9113924050632911, 'mrr': 0.800548523206751}
boost=1.0: {'hit_rate': 0.9240506329113924, 'mrr': 0.8139240506329113}
boost=3.0: {'hit_rate': 0.8987341772151899, 'mrr': 0.7693248945147676}
boost=5.0: {'hit_rate': 0.8708860759493671, 'mrr': 0.7401265822784809}
boost=10.0: {'hit_rate': 0.8582278481012658, 'mrr': 0.7122362869198313}
```

Increasing the question boost makes the metrics worse, not better. The
best value here is `1.0`, no boost at all. That's already the opposite of
what the intuition predicted.

But this is only one parameter. We can also tune `answer` and `section`
together with `question`.

> Consider a library like `Hyperopt` to make the same 

Define a search function with all three boosts:


In [79]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Now do a small grid search:

In [80]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/515 [00:00<?, ?it/s]

In [81]:
results

[{'question': 1.0,
  'answer': 1.0,
  'section': 0.1,
  'hit_rate': 0.9262135922330097,
  'mrr': 0.8020711974110027},
 {'question': 1.0,
  'answer': 1.0,
  'section': 0.2,
  'hit_rate': 0.9242718446601942,
  'mrr': 0.8052103559870547},
 {'question': 1.0,
  'answer': 1.0,
  'section': 0.5,
  'hit_rate': 0.8970873786407767,
  'mrr': 0.7728802588996762},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.1,
  'hit_rate': 0.9650485436893204,
  'mrr': 0.8575404530744335},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.2,
  'hit_rate': 0.9611650485436893,
  'mrr': 0.8544983818770224},
 {'question': 1.0,
  'answer': 2.0,
  'section': 0.5,
  'hit_rate': 0.9514563106796117,
  'mrr': 0.8428478964401294},
 {'question': 1.0,
  'answer': 4.0,
  'section': 0.1,
  'hit_rate': 0.9650485436893204,
  'mrr': 0.8645307443365691},
 {'question': 1.0,
  'answer': 4.0,
  'section': 0.2,
  'hit_rate': 0.9650485436893204,
  'mrr': 0.8690614886731387},
 {'question': 1.0,
  'answer': 4.0,
  'section': 0.5,
  

Sort by MRR:


In [82]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.965049,0.869061
23,2.0,10.0,0.5,0.961165,0.866214
6,1.0,4.0,0.1,0.965049,0.864531
22,2.0,10.0,0.2,0.959223,0.862265
21,2.0,10.0,0.1,0.959223,0.861845
8,1.0,4.0,0.5,0.961165,0.859385
35,5.0,10.0,0.5,0.965049,0.857540
19,2.0,4.0,0.2,0.965049,0.857540
3,1.0,2.0,0.1,0.965049,0.857540
33,5.0,10.0,0.1,0.965049,0.856893


For the same data, the best rows are (from May 29, 2026):

```ts
question  answer  section  hit_rate  mrr
1.0       2.0     0.1      0.975     0.885
2.0       4.0     0.2      0.975     0.885
5.0       10.0    0.5      0.975     0.885
5.0       10.0    0.2      0.975     0.884
5.0       10.0    0.1      0.975     0.884
2.0       4.0     0.1      0.975     0.884
2.0       4.0     0.5      0.977     0.884
1.0       2.0     0.2      0.977     0.884
1.0       2.0     0.5      0.965     0.862
1.0       4.0     0.1      0.970     0.862
```

The best combination weights `answer` twice as heavily as `question`,
with almost no weight on `section`. So the data says the opposite of
where we started. The answer text matters more for retrieval than the
question text. The intuition was wrong, and we'd never have known without
measuring it. This is exactly why we evaluate instead of guess.

The first three rows have the same relative weights:

```text
question : answer : section = 1 : 2 : 0.1
```

So we can use the smaller and easier-to-read values:
`question=1.0`, `answer=2.0`, and `section=0.1`. This gives the same
relative weights as `question=5.0`, `answer=10.0`, and `section=0.5`,
but the numbers are not unnecessarily large.

Define the search function with these boosts:


In [83]:
def text_search(query): # from data MAY 29 / 2026
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [84]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.9650485436893204, 'mrr': 0.8575404530744335}

Usually we care about both metrics. Hit Rate tells us whether the
correct document appears at all. MRR tells us whether it appears near
the top. A document near the top is more likely to be used by the RAG
prompt.

## Tuning Workflow

Search parameters can look arbitrary. This includes field boosts,
number of results, filters, and other settings. Evaluation gives us a
way to compare settings with evidence.

Grid search is fine when there are only a few settings. For a larger
parameter space, use a smarter search strategy. You can sample random
combinations, use Bayesian optimization, or keep a validation split so
you don't overfit the evaluation set.

For text search on our dataset, grid search takes about one second per
combination. That makes it practical to try many options. When each
evaluation takes minutes instead of seconds, grid search becomes too
expensive. In those cases, use Bayesian optimization with a library like
hyperopt. It explores the parameter space more efficiently by focusing
on combinations that are likely to improve the metric.

## Top-K tradeoffs

We return 5 results from search. Increasing top-K to 10 would improve
hit rate because there are more chances to find the correct document.
But more results means more context sent to the LLM. That costs more
and makes it harder for the model to identify what is relevant. Five
results is a reasonable default for short FAQ-style documents.

Next, we'll move from retrieval quality to answer quality and evaluate
the full RAG pipeline.